# Save triplets (template, science, difference) images (or cutouts or stamps or images) for each object identified and saved in `0.alerce_get_vsandbogus_objs.ipynb` and  `01.alerce_get_sn_objs.ipynb`

The stamps are in `.fits` format, this notebook shows how to get the triplet stamps with the object id, open it, convert to `.npy` format (numpy array) and save it.

## This notebook is a demonstration of how to save the stamps using the broker Fink. If, for the future, you want to save more stamps, I would recomend doing it in your computer or an external computer. Colab notebook sessions die fast.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor
import glob

import io
import requests
from astropy.io import fits

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Read csv file saved in previous step, to get the object ids

In [ ]:
path_save = "/content/drive/MyDrive/hackathon_colombia_oct_2026"

In [ ]:
df_all = pd.read_csv(os.path.join(path_save,'alerce_vs_bogus_classifier2_probgt08.csv.gz'))

In [ ]:
df_all['oid'].unique().shape

(202441,)

In [ ]:
df_all.shape

(202441, 6)

# Define where you want to save the images

In [ ]:
path_save = "/content/drive/MyDrive/hackathon_colombia_oct_2026/hack_colombia_stamps_oct_26"

In [ ]:
df_all.head()

,oid,ra,dec,ndiasources,class_name,probability
0,170591539567788216,354.812457,2.683815,3,bogus,0.847327
1,170631095481009094,299.543671,-22.278405,2,bogus,0.938549
2,170631094439248199,296.893579,-27.960011,2,bogus,0.958741
3,170503520713180759,253.163192,-20.319174,2,bogus,0.882257
4,170591539905429520,347.780913,-3.278336,2,bogus,0.860402


# Demonstration only for vs objects

In [ ]:
oid_list_vs = df_all.query('class_name == "VS"')['oid'].tolist()

In [ ]:
df_all.query('class_name == "VS"')['oid'].shape

(21710,)

# Everytime you run this, check which images you already saved

In [ ]:
path_save = "/content/drive/MyDrive/hackathon_colombia_oct_2026/hack_colombia_stamps_oct_26/VS"
# list of all paths in the path_save directory
list_save = glob.glob(f"{path_save}/*/*diff*npy")
ids_ready = [int(x.split('/')[-1].split('_')[0]) for x in list_save]
oid_list_vs = [x for x in oid_list_vs if x not in ids_ready]

In [ ]:
len(oid_list_vs)

35

In [ ]:


# Base URL for the Fink LSST API
API_URL = "https://api.lsst.fink-portal.org/api/v1/cutouts"


# Cutout types available from Rubin LSST: 'Science', 'Template', or 'Difference'
stamp_kinds = ["Science", "Template", "Difference"]

for oid in oid_list_vs[:10]:

    # Step 1: Get the list of diaSourceId for the object
    sources_url = "https://api.lsst.fink-portal.org/api/v1/sources"
    payload = {
        "diaObjectId": str(oid),
        "columns": "r:diaSourceId,r:midpointMjdTai",
    }

    response = requests.post(sources_url, json=payload)

    if response.status_code == 200:
        try:
          target_source_id = response.json()[-1]["r:diaSourceId"]
          print(f"Target diaSourceId found: {target_source_id}")
        except:
          print(f"No diaSourceId found for {oid}")
          continue
    else:
        print(f"Error fetching sources: {response.status_code}")



    for kind in stamp_kinds:
        print(f"Downloading {kind} stamp for {oid}...")

        # Construct payload
        payload = {
            "diaSourceId": str(target_source_id),
            "kind": kind,
            "output-format": "FITS"  # You can also use 'PNG' if preferred
        }

        os.makedirs(os.path.join(path_save,f"{oid}"), exist_ok=True)

        try:
            # Send POST request to Fink API
            response = requests.post(API_URL, json=payload)

            if response.status_code == 200 and response.content:

              # Read FITS bytes directly from memory
              fits_bytes = io.BytesIO(response.content)

              with fits.open(fits_bytes, ignore_missing_simple=True) as hdul:
                  # Extract image matrix as a NumPy array (typically extension 0 or 1 depending on layout)
                  img_data = hdul[0].data

                  # In case the primary extension is empty (header-only), fall back to extension 1
                  if img_data is None and len(hdul) > 1:
                      img_data = hdul[1].data

                  if img_data is not None:
                      # Define output file path

                      # Save as a standard binary numpy file

                      np.save(os.path.join(path_save,f"{oid}/{oid}_{kind.lower()}_cutout.npy"), img_data)
                      print(f" Successfully saved: {oid} (Shape: {img_data.shape})")
                  else:
                      print(f" Warning: No array data found in FITS for ID {oid}")

            else:
                print(f"Failed to fetch {kind} for {oid}: HTTP {response.status_code}")

        except Exception as e:
            print(f"An error occurred while fetching {oid}: {e}")


Streaming output truncated to the last 5000 lines.
 Successfully saved: 170591545044500687 (Shape: (30, 30))
Target diaSourceId found: 170525501694673700
 Successfully saved: 170525501694673700 (Shape: (37, 37))
 Successfully saved: 170525501694673700 (Shape: (37, 37))
 Successfully saved: 170525501694673700 (Shape: (37, 37))
Target diaSourceId found: 170587116515360952
 Successfully saved: 170587116515360952 (Shape: (31, 31))
 Successfully saved: 170587116515360952 (Shape: (31, 31))
 Successfully saved: 170587116515360952 (Shape: (31, 31))
Target diaSourceId found: 170635511950475454
 Successfully saved: 170635511950475454 (Shape: (31, 31))
 Successfully saved: 170635511950475454 (Shape: (31, 31))
 Successfully saved: 170635511950475454 (Shape: (31, 31))
Target diaSourceId found: 170587115080909155
 Successfully saved: 170587115080909155 (Shape: (30, 30))
 Successfully saved: 170587115080909155 (Shape: (30, 30))
 Successfully saved: 170587115080909155 (Shape: (30, 30))
Target diaSourc